### Imports + robust repo root + DB path (fixes your path errors permanently)

In [1]:
from pathlib import Path

def find_repo_root(start=None, max_up=10):
    p = (start or Path.cwd()).resolve()
    for _ in range(max_up):
        if (p / "Day-11" / "data" / "warehouse").exists():
            return p
        if (p / ".git").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Could not find repo root (expected to see Day-11/data/warehouse somewhere above).")

REPO_ROOT = find_repo_root()
DAY20_DIR = REPO_ROOT / "Day-20"
REPORTS = DAY20_DIR / "reports"
ARTIFACTS = DAY20_DIR / "artifacts"
SRC = REPO_ROOT / "src"

DB_PATH = REPO_ROOT / "Day-11" / "data" / "warehouse" / "day11_noshow.duckdb"

REPORTS.mkdir(parents=True, exist_ok=True)
ARTIFACTS.mkdir(parents=True, exist_ok=True)
SRC.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("DB_PATH:", DB_PATH)


REPO_ROOT: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science
DB_PATH: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-11\data\warehouse\day11_noshow.duckdb


### Create environment files (pip + conda)

In [2]:
requirements_txt = """\
duckdb>=1.0.0
pandas>=2.0.0
numpy>=1.24.0
scikit-learn>=1.4.0
matplotlib>=3.7.0
"""

env_yml = """\
name: noshow-pipeline
channels:
  - conda-forge
dependencies:
  - python=3.11
  - pip
  - pip:
      - duckdb>=1.0.0
      - pandas>=2.0.0
      - numpy>=1.24.0
      - scikit-learn>=1.4.0
      - matplotlib>=3.7.0
"""

(REPO_ROOT / "requirements.txt").write_text(requirements_txt, encoding="utf-8")
(REPO_ROOT / "environment.yml").write_text(env_yml, encoding="utf-8")

print("Wrote:", REPO_ROOT / "requirements.txt")
print("Wrote:", REPO_ROOT / "environment.yml")


Wrote: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\requirements.txt
Wrote: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\environment.yml


### Write the “one command” pipeline script (Day 18–19 logic, production-safe)

This script:

- reads gold_appointments_features_v1

- ensures splits exist (creates gold_appointments_splits if missing)

- fits propensity + two outcome models (mu0, mu1)

- computes uplift and risk policies for a budget

- writes policy_runs and policy_recommendations to DuckDB

- saves decisions_test.csv + policy_summary.csv to Day-20/reports/

In [3]:
pipeline_py = r'''\
import argparse
import uuid
from datetime import datetime
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression


def ensure_splits(con, features_tbl, splits_tbl="gold_appointments_splits", seed=42):
    tables = {t[0] for t in con.execute("SHOW TABLES").fetchall()}
    if splits_tbl in tables:
        return splits_tbl

    # patient-level split on person_id (train/valid/test)
    base = con.execute(f"SELECT DISTINCT person_id FROM {features_tbl} WHERE person_id IS NOT NULL").df()
    people = np.sort(base["person_id"].unique())
    rng = np.random.default_rng(seed)
    rng.shuffle(people)

    n = len(people)
    n_train = int(0.70 * n)
    n_valid = int(0.15 * n)
    train_ids = set(people[:n_train])
    valid_ids = set(people[n_train:n_train+n_valid])
    test_ids  = set(people[n_train+n_valid:])

    df = con.execute(f"SELECT appointment_id, person_id FROM {features_tbl}").df()
    def assign(pid):
        if pid in train_ids: return "train"
        if pid in valid_ids: return "valid"
        return "test"

    splits = df.copy()
    splits["split"] = splits["person_id"].map(assign)

    con.register("splits_df", splits)
    con.execute(f"CREATE OR REPLACE TABLE {splits_tbl} AS SELECT * FROM splits_df")
    return splits_tbl


def build_preprocess(df, X_cols):
    X = df[X_cols].copy()
    cat_cols = [c for c in X_cols if str(X[c].dtype) == "object"]
    num_cols = [c for c in X_cols if c not in cat_cols]

    prep = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), num_cols),
            ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                              ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
        ],
        remainder="drop"
    )
    return prep


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--db-path", type=str, required=True)
    ap.add_argument("--features-tbl", type=str, default="gold_appointments_features_v1")
    ap.add_argument("--splits-tbl", type=str, default="gold_appointments_splits")
    ap.add_argument("--budget-frac", type=float, default=0.10)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--outdir", type=str, required=True)
    args = ap.parse_args()

    db_path = Path(args.db_path).resolve()
    outdir = Path(args.outdir).resolve()
    outdir.mkdir(parents=True, exist_ok=True)

    # IMPORTANT: only one connection (avoids your “file is used by another process” errors)
    con = duckdb.connect(str(db_path))

    # ensure splits exist
    splits_tbl = ensure_splits(con, args.features_tbl, args.splits_tbl, seed=args.seed)

    df = con.execute(f"""
        SELECT f.*, s.split
        FROM {args.features_tbl} f
        LEFT JOIN {splits_tbl} s
        USING(appointment_id, person_id)
    """).df()

    # core columns
    A_COL = "sms_received"
    Y_COL = "label"

    # choose covariates (exclude ids, treatment, outcome, split)
    drop_cols = {"appointment_id", "person_id", A_COL, Y_COL, "split", "appt_date", "sched_date"}
    X_cols = [c for c in df.columns if c not in drop_cols]

    # basic cleanup
    df = df[df["split"].isin(["train","valid","test"])].copy()
    df[A_COL] = df[A_COL].astype(int)
    df[Y_COL] = df[Y_COL].astype(int)

    m_tr = df["split"].eq("train").to_numpy()
    m_va = df["split"].eq("valid").to_numpy()
    m_te = df["split"].eq("test").to_numpy()

    A = df[A_COL].to_numpy()
    Y = df[Y_COL].to_numpy()

    # preprocess template (CLONE IT for each pipeline to avoid the feature-mismatch bug you hit earlier)
    prep_template = build_preprocess(df, X_cols)

    # propensity model (train->valid AUC)
    ps_pipe = Pipeline([
        ("prep", clone(prep_template)),
        ("clf", LogisticRegression(solver="saga", max_iter=5000))
    ])
    ps_pipe.fit(df.loc[m_tr, X_cols], A[m_tr])
    ps_va = ps_pipe.predict_proba(df.loc[m_va, X_cols])[:, 1]
    ps_auc = roc_auc_score(A[m_va], ps_va)

    # outcome models (separate models for treated and control)
    y1_pipe = Pipeline([
        ("prep", clone(prep_template)),
        ("clf", LogisticRegression(solver="saga", max_iter=5000))
    ])
    y0_pipe = Pipeline([
        ("prep", clone(prep_template)),
        ("clf", LogisticRegression(solver="saga", max_iter=5000))
    ])

    y1_pipe.fit(df.loc[m_tr & (A==1), X_cols], Y[m_tr & (A==1)])
    y0_pipe.fit(df.loc[m_tr & (A==0), X_cols], Y[m_tr & (A==0)])

    # potential outcomes on TEST
    X_te = df.loc[m_te, X_cols]
    mu1_te = y1_pipe.predict_proba(X_te)[:, 1]   # P(no-show | do SMS)
    mu0_te = y0_pipe.predict_proba(X_te)[:, 1]   # P(no-show | do no SMS)
    uplift_te = mu0_te - mu1_te                  # + means SMS helps (reduces no-show)

    df_te = df.loc[m_te, ["appointment_id","person_id","split"]].copy()
    df_te["mu0"] = mu0_te
    df_te["mu1"] = mu1_te
    df_te["uplift"] = uplift_te

    # budget
    n_te = len(df_te)
    K = int(round(args.budget_frac * n_te))
    K = max(K, 1)

    # risk policy: send SMS to highest baseline risk (mu0)
    df_te = df_te.sort_values("mu0", ascending=False).reset_index(drop=True)
    df_te["rank_risk"] = np.arange(1, n_te+1)
    df_te["send_sms_risk"] = (df_te["rank_risk"] <= K).astype(int)

    # uplift policy: send SMS to highest uplift
    df_u = df_te.sort_values("uplift", ascending=False).reset_index(drop=True)
    df_u["rank_uplift"] = np.arange(1, n_te+1)
    df_u["send_sms_uplift"] = (df_u["rank_uplift"] <= K).astype(int)
    df_u = df_u[["appointment_id","person_id","rank_uplift","send_sms_uplift"]]

    df_dec = df_te.merge(df_u, on=["appointment_id","person_id"], how="left")

    # expected no-show rate under policies
    none_rate = float(df_dec["mu0"].mean())
    all_rate  = float(df_dec["mu1"].mean())
    risk_rate = float(np.mean(np.where(df_dec["send_sms_risk"].to_numpy()==1, df_dec["mu1"], df_dec["mu0"])))
    uplift_rate = float(np.mean(np.where(df_dec["send_sms_uplift"].to_numpy()==1, df_dec["mu1"], df_dec["mu0"])))

    summary = pd.DataFrame([{
        "run_ts": datetime.now().isoformat(timespec="seconds"),
        "features_tbl": args.features_tbl,
        "splits_tbl": splits_tbl,
        "budget_frac": float(args.budget_frac),
        "budget_k": int(K),
        "ps_auc_valid": float(ps_auc),
        "no_show_rate_none": none_rate,
        "no_show_rate_all": all_rate,
        "no_show_rate_risk_policy": risk_rate,
        "no_show_rate_uplift_policy": uplift_rate,
        "reduction_vs_none_risk": none_rate - risk_rate,
        "reduction_vs_none_uplift": none_rate - uplift_rate,
    }])

    # save reports
    summary.to_csv(outdir / "DAY20_policy_summary.csv", index=False)
    df_dec.to_csv(outdir / "DAY20_decisions_test.csv", index=False)

    # write runs + recommendations to DuckDB (Day 19 schema)
    RUN_ID = str(uuid.uuid4())
    RUN_TS = datetime.now().isoformat(timespec="seconds")

    con.execute("""
    CREATE TABLE IF NOT EXISTS policy_runs (
        run_id VARCHAR,
        run_ts VARCHAR,
        features_table VARCHAR,
        splits_table VARCHAR,
        budget_k BIGINT,
        budget_frac DOUBLE,
        notes VARCHAR
    )
    """)

    runs_df = pd.DataFrame([{
        "run_id": RUN_ID,
        "run_ts": RUN_TS,
        "features_table": args.features_tbl,
        "splits_table": splits_tbl,
        "budget_k": int(K),
        "budget_frac": float(args.budget_frac),
        "notes": "Day20 packaged run: risk + uplift policies from mu0/mu1"
    }])

    con.register("runs_df", runs_df)
    con.execute("INSERT INTO policy_runs SELECT * FROM runs_df")

    con.execute("""
    CREATE TABLE IF NOT EXISTS policy_recommendations (
        run_id VARCHAR,
        strategy VARCHAR,
        budget_k BIGINT,
        appointment_id BIGINT,
        person_id BIGINT,
        split VARCHAR,
        mu0 DOUBLE,
        mu1 DOUBLE,
        uplift DOUBLE,
        send_sms INTEGER,
        rank BIGINT
    )
    """)

    # build recommendations long format (both strategies)
    base_cols = ["appointment_id","person_id","split","mu0","mu1","uplift"]

    rec_risk = df_dec[base_cols].copy()
    rec_risk["run_id"] = RUN_ID
    rec_risk["strategy"] = "risk"
    rec_risk["budget_k"] = int(K)
    rec_risk["send_sms"] = df_dec["send_sms_risk"].astype(int).to_numpy()
    rec_risk["rank"] = df_dec["rank_risk"].astype(int).to_numpy()

    rec_uplift = df_dec[base_cols].copy()
    rec_uplift["run_id"] = RUN_ID
    rec_uplift["strategy"] = "uplift"
    rec_uplift["budget_k"] = int(K)
    rec_uplift["send_sms"] = df_dec["send_sms_uplift"].astype(int).to_numpy()
    rec_uplift["rank"] = df_dec["rank_uplift"].astype(int).to_numpy()

    rec_all = pd.concat([rec_risk, rec_uplift], ignore_index=True)
    rec_all = rec_all[[
        "run_id","strategy","budget_k","appointment_id","person_id","split",
        "mu0","mu1","uplift","send_sms","rank"
    ]].copy()

    con.register("rec_all_df", rec_all)
    con.execute("INSERT INTO policy_recommendations SELECT * FROM rec_all_df")

    con.close()

    print("OK. Saved reports to:", outdir)
    print("RUN_ID:", RUN_ID)
    print(summary.to_string(index=False))


if __name__ == "__main__":
    main()
'''
(SRC / "run_project2_pipeline.py").write_text(pipeline_py, encoding="utf-8")
(SRC / "__init__.py").write_text("", encoding="utf-8")
print("Wrote:", SRC / "run_project2_pipeline.py")


Wrote: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\src\run_project2_pipeline.py


### Write DAY20.md (the end-of-project packaging report)

In [4]:
from datetime import datetime

day20_md = f"""\
# Day 20 — Packaging & Reproducible Pipeline (Project 2: No-Show + SMS)

Today we package Days 11–19 into a single reproducible pipeline that reads the DuckDB warehouse, ensures patient-level splits exist, estimates potential outcomes (mu0, mu1), produces risk-based and uplift-based SMS targeting lists under a budget, and writes policy outputs back into DuckDB.

## One command to run (from repo root)

Windows:
python -m src.run_project2_pipeline --db-path "{DB_PATH}" --outdir "Day-20/reports" --budget-frac 0.10

Alternative (relative DB path):
python -m src.run_project2_pipeline --db-path "Day-11/data/warehouse/day11_noshow.duckdb" --outdir "Day-20/reports" --budget-frac 0.10

## What gets produced

Files:
- Day-20/reports/DAY20_policy_summary.csv
- Day-20/reports/DAY20_decisions_test.csv

DuckDB tables:
- policy_runs
- policy_recommendations
- gold_appointments_splits (created if missing)

## Stability fixes included

We use a single DuckDB connection to avoid file-lock errors. The preprocessing objects are cloned per model to avoid one-hot feature mismatch.

Generated: {datetime.now().isoformat(timespec="seconds")}
"""

(REPORTS / "DAY20.md").write_text(day20_md, encoding="utf-8")

closeout = """\
# Project 2 Closeout — No-Show Prediction + SMS Causal/Uplift Targeting

This project built an end-to-end workflow that goes beyond prediction into causal estimation and decision deployment.

The predictive layer produces calibrated risk estimates for no-show, while the causal layer estimates the effect of SMS reminders on no-show (ATE) and then moves to heterogeneous effects (uplift), comparing two operational policies under a fixed SMS budget:
1) risk-based policy: target highest baseline risk
2) uplift-based policy: target those expected to benefit most from SMS

The Day-20 pipeline operationalizes the full workflow: it pulls gold features from DuckDB, enforces patient-level splits, trains models, generates mu0/mu1 and uplift on the test set, writes policy outputs into DuckDB as runs + recommendations, and exports test decisions and a policy summary CSV suitable for reporting or dashboards.
"""

(REPORTS / "PROJECT2_CLOSEOUT.md").write_text(closeout, encoding="utf-8")

print("Wrote:", REPORTS / "DAY20.md")
print("Wrote:", REPORTS / "PROJECT2_CLOSEOUT.md")


Wrote: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-20\reports\DAY20.md
Wrote: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-20\reports\PROJECT2_CLOSEOUT.md


### Writes run_day20.bat (no code fences)

In [5]:
bat = f"""@echo off
cd /d "{REPO_ROOT}"
python -m src.run_project2_pipeline --db-path "{DB_PATH}" --outdir "Day-20\\reports" --budget-frac 0.10
pause
"""
(REPO_ROOT / "run_day20.bat").write_text(bat, encoding="utf-8")
print("Wrote:", REPO_ROOT / "run_day20.bat")


Wrote: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\run_day20.bat
